In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("../data/distillation_purity_dataset.csv")

# Feature set (finalized after EDA + redundancy check)
features = ['R', 'N', 'z_F', 'alpha', 'F']
X = df[features]

y_xD = df['x_D']
y_QrebLog = np.log(df['Q_reb'])

print(X.shape, y_xD.shape, y_QrebLog.shape)
X.head()

(2000, 5) (2000,) (2000,)


,R,N,z_F,alpha,F
0,6.879788,18.962262,0.449968,1.719439,112.381165
1,1.942175,11.778706,0.284007,3.348684,164.909923
2,0.860352,12.388053,0.579089,2.880570,363.711317
3,2.108692,8.928213,0.364332,3.564057,141.873667
4,0.638938,15.333836,0.721173,2.963382,288.644323


In [2]:
X_train, X_test, y_xD_train, y_xD_test = train_test_split(
    X, y_xD, test_size=0.2, random_state=42
)

_, _, y_QrebLog_train, y_QrebLog_test = train_test_split(
    X, y_QrebLog, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(1600, 5) (400, 5)


In [3]:
lr_xD = LinearRegression()
lr_xD.fit(X_train, y_xD_train)

y_xD_pred = lr_xD.predict(X_test)

mse = mean_squared_error(y_xD_test, y_xD_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_xD_test, y_xD_pred)

print(f"Linear Regression - x_D prediction")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.4f}")

Linear Regression - x_D prediction
MSE:  0.000552
RMSE: 0.023487
R²:   0.2140


In [4]:
lr_Qreb = LinearRegression()
lr_Qreb.fit(X_train, y_QrebLog_train)

y_QrebLog_pred = lr_Qreb.predict(X_test)

mse = mean_squared_error(y_QrebLog_test, y_QrebLog_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_QrebLog_test, y_QrebLog_pred)

print(f"Linear Regression - log(Q_reb) prediction")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.4f}")

Linear Regression - log(Q_reb) prediction
MSE:  0.019645
RMSE: 0.140162
R²:   0.9529


In [5]:
rf_xD = RandomForestRegressor(n_estimators=200, random_state=42)
rf_xD.fit(X_train, y_xD_train)

y_xD_pred_rf = rf_xD.predict(X_test)

mse = mean_squared_error(y_xD_test, y_xD_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_xD_test, y_xD_pred_rf)

print(f"Random Forest - x_D prediction")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.4f}")

Random Forest - x_D prediction
MSE:  0.000295
RMSE: 0.017163
R²:   0.5803


In [6]:
features_with_xB = ['R', 'N', 'z_F', 'alpha', 'F', 'x_B']
X_xB = df[features_with_xB]

X_xB_train, X_xB_test, y_xD_train2, y_xD_test2 = train_test_split(
    X_xB, y_xD, test_size=0.2, random_state=42
)

rf_xD_v2 = RandomForestRegressor(n_estimators=200, random_state=42)
rf_xD_v2.fit(X_xB_train, y_xD_train2)

y_xD_pred_v2 = rf_xD_v2.predict(X_xB_test)

mse = mean_squared_error(y_xD_test2, y_xD_pred_v2)
rmse = np.sqrt(mse)
r2 = r2_score(y_xD_test2, y_xD_pred_v2)

print(f"Random Forest - x_D prediction (with x_B included)")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.4f}")

Random Forest - x_D prediction (with x_B included)
MSE:  0.000280
RMSE: 0.016728
R²:   0.6013


In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 2, 5]
}

rf_tune = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    rf_tune, param_grid, 
    cv=5, 
    scoring='r2', 
    n_jobs=-1
)

grid_search.fit(X_xB_train, y_xD_train2)

print("Best params:", grid_search.best_params_)
print("Best CV R²:", grid_search.best_score_)

# Evaluate best model on the held-out test set
best_rf = grid_search.best_estimator_
y_xD_pred_tuned = best_rf.predict(X_xB_test)

mse = mean_squared_error(y_xD_test2, y_xD_pred_tuned)
rmse = np.sqrt(mse)
r2 = r2_score(y_xD_test2, y_xD_pred_tuned)

print(f"\nTuned Random Forest - x_D prediction (test set)")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.4f}")

Best params: {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 500}
Best CV R²: 0.5622010706673459

Tuned Random Forest - x_D prediction (test set)
MSE:  0.000275
RMSE: 0.016590
R²:   0.6078


In [10]:
df_5000 = pd.read_csv("../data/distillation_purity_dataset.csv")

X_5000 = df_5000[['R', 'N', 'z_F', 'alpha', 'F', 'x_B']]
y_xD_5000 = df_5000['x_D']

X_5000_train, X_5000_test, y_5000_train, y_5000_test = train_test_split(
    X_5000, y_xD_5000, test_size=0.2, random_state=42
)

rf_5000 = RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=42)
rf_5000.fit(X_5000_train, y_5000_train)

y_5000_pred = rf_5000.predict(X_5000_test)

mse = mean_squared_error(y_5000_test, y_5000_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_5000_test, y_5000_pred)

print(f"Random Forest (tuned params) - x_D prediction, 5000 rows")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.4f}")

Random Forest (tuned params) - x_D prediction, 5000 rows
MSE:  0.000222
RMSE: 0.014885
R²:   0.7035


In [11]:
y_QrebLog_5000 = np.log(df_5000['Q_reb'])

X_Qreb_5000 = df_5000[['R', 'N', 'z_F', 'alpha', 'F']]  # x_B not relevant here, same as original Q_reb feature set

X_Qreb_train, X_Qreb_test, y_Qreb_train, y_Qreb_test = train_test_split(
    X_Qreb_5000, y_QrebLog_5000, test_size=0.2, random_state=42
)

rf_Qreb = RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=42)
rf_Qreb.fit(X_Qreb_train, y_Qreb_train)

y_Qreb_pred = rf_Qreb.predict(X_Qreb_test)

mse = mean_squared_error(y_Qreb_test, y_Qreb_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_Qreb_test, y_Qreb_pred)

print(f"Random Forest - log(Q_reb) prediction, 5000 rows")
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²:   {r2:.4f}")

Random Forest - log(Q_reb) prediction, 5000 rows
MSE:  0.002047
RMSE: 0.045243
R²:   0.9959
